# 🧠 FB PAGE BM AUTOMATED - LLM FINE-TUNING SUITE (UNSLOTH & LORa)
### Fine-Tune Llama-3.2 (3B) / Qwen-2.5 (3B) in Google Colab Free Tier (T4 GPU)
---
**Objective:** Fine-tune an ultra-fast, state-of-the-art Large Language Model (LLM) tailored for:
1. **Viral Facebook Ad Copywriting & Hook Generation** (Khmer & English)
2. **24/7 Automated Comment & Messenger Lead Conversion Agent**
3. **Zero-Ban Meta Advertising Policy Compliance Guard**

**Hardware Target:** Google Colab Free Tier (1x Tesla T4 GPU, 15GB VRAM)  
**Technology:** Unsloth (5X faster training, 80% memory reduction, 4-bit QLoRA)

## 📦 Step 1: Install Unsloth & Modern Dependencies

In [ ]:
%%capture
!pip install --quiet unsloth unsloth_zoo peft bitsandbytes trl accelerate datasets transformers huggingface_hub

import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"⚡ GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 🚀 Step 2: Load Pre-Trained 4-bit Quantized Model (Llama-3.2 3B or Qwen-2.5 3B)
We use 4-bit quantization to fit comfortably within 6GB of VRAM on the free Colab T4.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto detection (Float16 for T4)
load_in_4bit = True

# Choose model: 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit' or 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print(f"✅ Loaded {model_name} successfully into VRAM!")

## ⚙️ Step 3: Configure PEFT / LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Suggested 8, 16, 32
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print("✅ LoRA trainable parameters initialized!")

## 📊 Step 4: Prepare High-Converting Facebook Marketing & Support Dataset (Khmer & English)

In [ ]:
from datasets import Dataset

# Prompt template for instruction tuning
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Training dataset of high-converting FB ad hooks & customer replies
training_samples = [
    {
        "instruction": "Create a high-converting Facebook Ad caption in Khmer with a powerful curiosity hook.",
        "input": "Product: Angkor Quant AI Trading Assistant. Target: Retail traders wanting to avoid liquidation.",
        "output": "🔥 កំពូលយុទ្ធសាស្ត្រ Quant Trading ជំនាន់ថ្មី 2026! 🚀\nតើអ្នកធ្លាប់ឆ្ងល់ទេ ហេតុអ្វីបានជា 90% នៃ Trader តែងតែបាត់បង់ដើមទុនក្នុងពេលទីផ្សារបោក?\nឈប់ដេញចាប់តម្លៃតាមអារម្មណ៍! ប្រព័ន្ធ Angkor Quant AI បំពាក់នូវ Breakeven Armor និង 360° Trap Clearance ការពារដើមទុន ១០០%។ 💡💎\n👉 ចុចទីនេះដើម្បីទទួលបានដំណោះស្រាយភ្លាមៗ!"
    },
    {
        "instruction": "Auto-respond to a customer comment inquiring about price on a Facebook Boosted Post.",
        "input": "Comment: តម្លៃប៉ុន្មានបង? ចង់ទិញ",
        "output": "សួស្តីបង! អរគុណសម្រាប់ការចាប់អារម្មណ៍ 🙏 ខ្ញុំបានផ្ញើព័ត៌មានលម្អិត តម្លៃ និងប្រូម៉ូសិនពិសេសជូនក្នុង Messenger Inbox រួចរាល់ហើយបាទ! សូមបងពិនិត្យមើលប្រអប់សារឥឡូវនេះបាន 📩✨"
    },
    {
        "instruction": "Generate a pain-point direct-response Facebook ad copy in English.",
        "input": "Product: E-commerce Automation Suite. Target: Store owners tired of burning ad budget.",
        "output": "⚠️ Still burning thousands on Facebook Ads with 0.8X ROAS? Stop throwing money at cold audiences!\nOur Autonomous AI pacer tests creative hooks with $5 test budgets, automatically scales winners to 4.5X ROAS, and kills losing ad sets before you lose a single dollar. 📈⚡\n👉 Click the link below to get your free 14-day case study!"
    },
    {
        "instruction": "Classify if this Facebook ad copy violates Meta Advertising Standards.",
        "input": "Copy: Get rich overnight! 100% guaranteed profit in 24 hours without any risk!",
        "output": "VIOLATION_DETECTED: Misleading Financial Claims & Guaranteed Profits Policy. Immediate account ban risk. Recommendation: Rephrase to focus on technology, systematic backtesting, and risk management."
    }
]

# Format into dataset
formatted_data = []
for s in training_samples * 150:  # Expand to 600 training pairs
    text = alpaca_prompt.format(s["instruction"], s["input"], s["output"]) + tokenizer.eos_token
    formatted_data.append({"text": text})

dataset = Dataset.from_list(formatted_data)
print(f"✅ Prepared {len(dataset)} instruction fine-tuning records!")

## 🏋️ Step 5: Train Model with SFTTrainer (Supervised Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

supports_bf16 = is_bfloat16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not supports_bf16,
        bf16=supports_bf16,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

print("⚡ Initiating fine-tuning on Colab GPU...")
trainer_stats = trainer.train()
print("🎉 Training completed successfully!")

## 🧪 Step 6: Test Inference Live

In [ ]:
FastLanguageModel.for_inference(model)

prompt = alpaca_prompt.format(
    "Create a viral Facebook Ad caption in Khmer with a powerful curiosity hook.",
    "Product: Gold Trading Algorithm with 90% Win Rate",
    ""
)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## 💾 Step 7: Export LoRA Adapter & GGUF (for VPS or Local Telegram Bot)

In [ ]:
# Save LoRA adapter locally or to Google Drive
model.save_pretrained("fb_ad_llm_lora")
tokenizer.save_pretrained("fb_ad_llm_lora")

# Optional: Export to 4-bit GGUF for llama.cpp on Linux VPS
# model.save_pretrained_gguf("fb_ad_llm_gguf", tokenizer, quantization_method="q4_k_m")

print("✅ Fine-tuned weights saved to 'fb_ad_llm_lora/'!")
print("🚀 You can now copy these weights or push to HuggingFace Hub for instant use with BM AUTOMATED!")